# Strategic Market Intelligence Pipeline
> **Reinsurance Data Lake** - Connector Pattern + DuckDB Medallion Lake + GPT-4o LLM Extraction

This notebook pulls all pipeline modules from the GitHub Gist and runs the full ingestion flow.

**Steps:**
1. Download source files from Gist
2. Install dependencies
3. Configure API keys
4. Run the pipeline

## Step 1 - Download source files from GitHub Gist

In [ ]:
import urllib.request, json, os

GIST_ID = "19941613e9b7ee9b92e630468ab6c65d"
GIST_API = f"https://api.github.com/gists/{GIST_ID}"

with urllib.request.urlopen(GIST_API) as r:
    gist = json.loads(r.read())

files_written = []
for filename, meta in gist["files"].items():
    raw_url = meta["raw_url"]
    with urllib.request.urlopen(raw_url) as r:
        content = r.read().decode("utf-8")
    with open(filename, "w") as f:
        f.write(content)
    files_written.append(filename)
    print(f"  downloaded: {filename} ({len(content):,} chars)")

print(f"\nDone - {len(files_written)} files downloaded.")

## Step 2 - Install dependencies

In [ ]:
%pip install -q -r requirements.txt
print("All dependencies installed.")

## Step 3 - Configure API keys

**Option A (recommended):** Use Colab Secrets (lock icon in left sidebar)
- Add a secret named `OPENAI_API_KEY` with your key value

**Option B:** Paste directly below (never share this notebook publicly if you do this)

In [ ]:
import os

# --- Option A: Colab Secrets (recommended) ---
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OpenAI key loaded from Colab Secrets.")
except Exception:
    # --- Option B: Paste your key here ---
    os.environ["OPENAI_API_KEY"] = "sk-..."   # <-- replace with your key
    print("OpenAI key set manually.")

# Optional: override the model
os.environ["OPENAI_MODEL"] = "gpt-4o-mini"   # or gpt-4o for higher quality
os.environ["LOG_LEVEL"] = "INFO"

## Step 4 - Initialise the Data Lake

In [ ]:
from database import initialise_lake, lake_stats
initialise_lake()
print("Medallion Data Lake ready.")

## Step 5 - Run: Ingest RSS feeds -> Bronze layer

In [ ]:
from connectors import RSSConnector
from database import bulk_insert_bronze_records

connector = RSSConnector()
connector.authenticate()

articles = list(connector.fetch())
connector.close()

records = [
    {
        "source_name": a.source_name,
        "raw_text": a.raw_text,
        "source_url": a.source_url,
        "title": a.title,
        "raw_metadata": a.raw_metadata,
    }
    for a in articles
]

ids = bulk_insert_bronze_records(records)
print(f"Ingested {len(articles)} articles -> Bronze layer ({len(ids)} records saved).")

## Step 6 - Run: LLM Extraction -> Silver layer

> Requires a valid `OPENAI_API_KEY`. Skip this cell for a dry-run.

In [ ]:
from database import fetch_unprocessed_bronze, insert_silver_signal, mark_bronze_processed
from processor import IntelligenceProcessor

processor = IntelligenceProcessor()
bronze_records = fetch_unprocessed_bronze(limit=20)  # increase limit as needed
print(f"Processing {len(bronze_records)} Bronze records...")

silver_signals = processor.extract_batch(bronze_records)

for signal in silver_signals:
    insert_silver_signal(signal)
    mark_bronze_processed(signal["bronze_id"])

# Mark noise records as processed too
silver_ids = {s["bronze_id"] for s in silver_signals}
for rec in bronze_records:
    if rec["id"] not in silver_ids:
        mark_bronze_processed(rec["id"])

print(f"Done - {len(silver_signals)} signals saved to Silver layer.")

## Step 7 - Explore the results

In [ ]:
import duckdb
from config import settings

con = duckdb.connect(str(settings.duckdb_path))

print("=== GOLD LAYER SUMMARY ===")
display(con.execute("SELECT * FROM gold_signal_summary").df())

In [ ]:
print("=== SILVER SIGNALS ===")
display(con.execute("""
    SELECT
        signal_category,
        title,
        summary,
        sentiment,
        ROUND(confidence_score, 2) AS confidence,
        action_required,
        extracted_at
    FROM silver_intelligence_signals
    ORDER BY extracted_at DESC
""").df())

In [ ]:
print("=== ACTION ITEMS (human review required) ===")
display(con.execute("""
    SELECT signal_category, title, summary, source_url
    FROM silver_intelligence_signals
    WHERE action_required = TRUE
    ORDER BY extracted_at DESC
""").df())

In [ ]:
# Filter by category
CATEGORY = "Emerging_Risks"  # Competitor_Strategy | Broker_Dynamics | Emerging_Risks

display(con.execute(f"""
    SELECT title, summary, key_entities, confidence_score
    FROM silver_intelligence_signals
    WHERE signal_category = '{CATEGORY}'
    ORDER BY confidence_score DESC
""").df())